In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from itertools import combinations
import re

# ================= LOAD =================
df = pd.read_csv("data/vn_music_simplification.csv")

# ================= FEATURE ENGINEERING =================
df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
df['hour'] = df['published_at'].dt.hour
df['day_of_week'] = df['published_at'].dt.dayofweek

# convert numeric
num_cols = [
    'view_count','like_count','comment_count',
    'tags_count',
    'channel_subscriber_count',
    'channel_view_count',
    'channel_video_count'
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')


def parse_duration_iso8601(duration):
    if pd.isna(duration):
        return np.nan

    if isinstance(duration, (int, float, np.integer, np.floating)):
        return float(duration)

    duration = str(duration).strip().upper()
    pattern = r'^P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?(?:(\d+(?:\.\d+)?)S)?)?$'
    match = re.fullmatch(pattern, duration)

    if not match:
        return np.nan

    days = int(match.group(1)) if match.group(1) else 0
    hours = int(match.group(2)) if match.group(2) else 0
    minutes = int(match.group(3)) if match.group(3) else 0
    seconds = float(match.group(4)) if match.group(4) else 0

    return days * 86400 + hours * 3600 + minutes * 60 + seconds

# parse duration before cleaning/modeling
df['duration_sec'] = df['duration'].apply(parse_duration_iso8601)

# ================= CLEAN =================
df = df.replace([np.inf, -np.inf], np.nan)

# log target (IMPORTANT)
df['log_view'] = np.log1p(df['view_count'])

# ================= SELECT FEATURES =================
features = [
    'duration_sec',
    'tags_count',
    'channel_subscriber_count',
    'channel_view_count',
    'channel_video_count',
    'hour',
    'day_of_week'
]

# drop NA
df_model = df.dropna(subset=features + ['log_view']).copy()

print("Data size for modeling:", len(df_model))

# ================= FUNCTION =================
def check_confounder(df, X, Z, y='log_view'):
    """
    X: variable of interest
    Z: potential confounder
    """

    try:
        # Model 1: naive
        X1 = sm.add_constant(df[[X]])
        m1 = sm.OLS(df[y], X1).fit()
        coef1 = m1.params[X]

        # Model 2: with confounder
        X2 = sm.add_constant(df[[X, Z]])
        m2 = sm.OLS(df[y], X2).fit()
        coef2 = m2.params[X]

        # % change
        change = (coef2 - coef1) / (abs(coef1) + 1e-6)

        return coef1, coef2, change

    except Exception:
        return None

# ================= SCAN ALL PAIRS =================
results = []

for X, Z in combinations(features, 2):
    res = check_confounder(df_model, X, Z)
    if res:
        coef1, coef2, change = res

        results.append({
            'X': X,
            'Z (potential confounder)': Z,
            'coef_before': coef1,
            'coef_after': coef2,
            'change_%': change * 100
        })

        # check reverse direction also
        res_rev = check_confounder(df_model, Z, X)
        if res_rev:
            coef1_r, coef2_r, change_r = res_rev
            results.append({
                'X': Z,
                'Z (potential confounder)': X,
                'coef_before': coef1_r,
                'coef_after': coef2_r,
                'change_%': change_r * 100
            })

# ================= RESULT =================
result_df = pd.DataFrame(results)

if result_df.empty:
    print("No valid confounder pairs were found.")
else:
    # sort theo mức thay đổi mạnh nhất
    result_df = result_df.sort_values(by='change_%', key=lambda x: abs(x), ascending=False)

    print("\nTop potential confounders:")
    print(result_df.head(20))

    # save file
    result_df.to_csv("confounder_scan_results.csv", index=False)

Data size for modeling: 0
No valid confounder pairs were found.


In [8]:
print("\n===== RAW DATA =====")
print("Total rows:", len(df))

print("\nMissing values per column:")
print(df.isna().sum())


===== RAW DATA =====
Total rows: 10615

Missing values per column:
video_id                        0
published_at                    0
channel_id                      0
channel_title                   0
title                           0
description                  1079
tags                         1890
category_id                     0
view_count                      0
like_count                    597
comment_count                 164
duration                    10615
definition                      0
caption                         0
licensed_content                0
privacy_status                  0
made_for_kids                   0
tags_count                      0
topic_categories               18
search_query                    0
channel_subscriber_count        2
channel_view_count              2
channel_video_count             2
hour                            0
day_of_week                     0
log_view                        0
duration_sec                10615
dtype: int64
